In [1]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from dataclasses import replace
import importlib

import run_config
import utils_lya_halo

from run_config import cfg, smoke
from utils_lya_halo import run_extract, read_galaxy_fits, run_stack, measure, plotting
from utils_lya_halo import PipelineConfig
from utils_lya_halo import starpsf
from astroquery.gaia import Gaia

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).


In [2]:
#Pre-test
!python test_framework.py

[1] config, provenance, derived paths OK: {'FIELD': 'AEGIS', 'BINMODE': 'virial', 'COMBINE': 'biweight'}
[2] FITS round-trip + alignment OK: shape=(5, 6, 64), ngal==len(catalog)==5
[3a] Stage 2 OK: 4 methods, cube (6, 6, 161), VR_biweight_v=79.55349685569028
100%|███████████████████████████████████████████| 5/5 [00:00<00:00, 1559.34it/s]
[3b] Stage 3 (run_measure) OK: centroid_v_med shape (6,), per-pixel err (6, 161)

ALL SMOKE CHECKS PASSED


In [3]:
# #Smoke test with N galaxies
# N_smoke = 3
# file_name = run_extract(smoke(N_smoke))

In [4]:
#Run in terminal to see current memory usage
#   watch -n1 cat /sys/fs/cgroup/memory.current

# Star Positions

In [5]:
field_select = 'AEGIS'

In [6]:
star_mag_min, star_mag_max = 16, 23

if (field_select == 'AEGIS'):
    ra_min, ra_max = 214.6, 215.2
    dec_min, dec_max = 52.7, 53.0
elif (field_select == 'COSMOS'):
    ra_min, ra_max = 149.9, 150.3
    dec_min, dec_max = 2.1, 2.5
else:
    print('Did not set bounds, specify field using either COSMOS or AEGIS')

query = f"""
SELECT
    ra, dec, phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE ra BETWEEN {ra_min} AND {ra_max}
AND dec BETWEEN {dec_min} AND {dec_max}
AND parallax_over_error > 5
AND phot_g_mean_mag BETWEEN {star_mag_min} AND {star_mag_max}
"""

job = Gaia.launch_job(query)
stars = job.get_results()

In [7]:
stars

ra,dec,phot_g_mean_mag
deg,deg,mag
float64,float64,float32
214.6295136433056,52.75903025101095,18.299902
215.05637237994242,52.78921589740662,17.031715
214.93076591100538,52.8028357682429,16.261246
214.86583943583864,52.87341894083904,16.771942
214.81967853970315,52.975222734676464,16.37511
214.99133960842218,52.78366936474037,18.889494
214.61398490650095,52.81074564917017,19.351244
214.6955385155793,52.923331652323235,18.470182


# Extracting the star data

In [11]:
# 1. config binned in arcsec, separate catalog/output_dir to avoid cache/FITS collisions
star_cfg = replace(
    cfg,
    field=field_select,
    catalog="STAR_PSF",
    output_dir="./outputs_starpsf",
    bin_mode="arcsec",
    bins=[0, 1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 40, 50, 70, 100])

# 2. Star coordinates (deg). z/mass are dummies in arcsec mode.
star_ra  = stars['ra']
star_dec = stars['dec']
star_tbl = starpsf.make_star_run_table(star_cfg, star_ra, star_dec, mode="arcsec")

# 3. Extract — mask='none' keeps the bright star core. (the one expensive step)
star_prod = starpsf.run_positions_extract(star_cfg, star_tbl,
    mask="none",
    use_cache=True,
    write=True)

Loaded AEGIS: 141 exposures, 34944 fibers, 1036 wavelength pixels


100%|██████████| 88/88 [2:21:10<00:00, 96.26s/it]   

wrote ./outputs_starpsf/galaxy_catc1488c_b14_2dbbb7_AEGIS_STAR_PSF_arcsec_biweight_image_bg57-63.fits
extracted 88 positions, mask='none', bin_mode=arcsec


# Extracting the random position data

In [8]:
table_all = Table.read('catalogs/lya_halo_catalog_cuts/ALL_haoiii_sn5_mosdef_catalog.txt', format='ascii')

gal_run_table = table_all[table_all['FIELD'] == field_select]
table_len = len(gal_run_table)

print(table_len)
gal_run_table[:3]

248


INDEX,FIELD,CATALOG,RA,DEC,z,MASS_16,MASS_50,MASS_84,SFR100_16,SFR100_50,SFR100_84,SFR10_16,SFR10_50,SFR10_84,E(B-V)_16,E(B-V)_50,E(B-V)_84,HA_FLUX,HA_FLUX_ERR,HA_SN,HA_FLAG,HA_FWHM,HA_FWHM_ERR,HB_FLUX,HB_FLUX_ERR,HB_SN,HB_FLAG,HB_FWHM,HB_FWHM_ERR,OII_FLUX,OII_FLUX_ERR,OII_SN,OII_FLAG,OII3727_FLUX,OII3727_FLUX_ERR,OII3727_SN,OII3727_FLAG,OII3727_FWHM,OII3727_FWHM_ERR,OII3730_FLUX,OII3730_FLUX_ERR,OII3730_SN,OII3730_FLAG,OII3730_FWHM,OII3730_FWHM_ERR,OIII_FLUX,OIII_FLUX_ERR,OIII_SN,OIII_FLAG,OIII4960_FLUX,OIII4960_FLUX_ERR,OIII4960_SN,OIII4960_FLAG,OIII4960_FWHM,OIII4960_FWHM_ERR,OIII5008_FLUX,OIII5008_FLUX_ERR,OIII5008_SN,OIII5008_FLAG,OIII5008_FWHM,OIII5008_FWHM_ERR,R814,R814_ERR,R160,R160_ERR,Z_16,Z_50,Z_84,BETA_16,BETA_50,BETA_84,AV_UV_16,AV_UV_50,AV_UV_84,O3,O32,E_BV_NEB_16,E_BV_NEB_50,E_BV_NEB_84,E_BV_STAR_TO_NEB_16,E_BV_STAR_TO_NEB_50,E_BV_STAR_TO_NEB_84,O3_ERR,O32_ERR
int64,str6,str6,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,float64,float64,float64,float64,float64,str5,float64,float64,float64,float64,float64,str5,float64,float64,float64,str5,float64,float64,float64,float64,float64,str5,float64,float64,float64,float64,float64,str5,float64,float64,float64,str5,float64,float64,float64,float64,float64,str5,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
988,AEGIS,MOSDEF,214.718475,52.798725,2.1307,9.989999771118164,10.039999961853027,10.039999961853027,1.0800000429153442,1.0800000429153442,1.3200000524520874,nan,nan,nan,0.1111111044883728,0.12345678359270096,0.17530862987041473,4.5544887e-17,4.3122119e-18,10.561838809451826,True,11.098027,0.84146168,1.4802864e-17,1.7398152e-18,8.508296743240317,True,11.501331,0.66662958,nan,nan,0.0,False,2.2303295e-17,6.0247431e-18,3.7019495486869807,True,7.5277491,1.5435472,2.5861069e-17,5.1973409e-18,4.9758269656700795,True,7.5333236,1.5446903,nan,nan,0.0,False,1.8471743e-17,1.6305244e-18,11.328713020179274,True,11.3784,0.69130917,4.6416571e-17,1.6048051e-18,28.923494198765944,True,4.6416571e-17,0.082212222,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.44999998807907104,0.5,0.7099999785423279,0.6418207408699157,0.12944066008996846,-0.05983008949491766,0.0547406991609075,0.20207924777432393,0.2525252374735745,0.28058359907432034,0.39842870425094257,0.053290847928758565,0.073361210652094
989,AEGIS,MOSDEF,214.737595,52.801903,2.2254,10.0,10.050000190734863,10.050000190734863,1.090000033378601,1.090000033378601,1.3300000429153442,nan,nan,nan,0.06172839179635048,0.07407407462596893,0.12345678359270096,1.0684958e-16,8.8163221e-18,12.1195186369155,True,11.176502,0.40223374,2.8263619e-17,2.3801611e-18,11.874666382876352,True,8.7444418,0.95555726,nan,nan,0.0,False,4.2779549e-17,8.4157919e-18,5.0832470085197805,True,7.1810833,0.90986627,3.4218304e-17,1.1319991e-17,3.0228207778610425,True,7.1863944,0.91055974,nan,nan,0.0,False,2.3820396e-17,1.8882964e-18,12.614754759898922,True,8.045026,0.72988313,8.4529722e-17,3.7386561e-18,22.6096543086699,True,8.4529722e-17,0.50308504,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.25,0.30000001192092896,0.5,0.58360161855168,0.14835077310987355,0.139476639662983,0.2357581303310737,0.34134582657027684,0.14029179953716017,0.16835016960447485,0.28058359907432034,0.04024238008640943,0.08131242364376463
990,AEGIS,MOSDEF,214.74292,52.799957,2.1328,9.949999809265137,10.029999732971191,10.069999694824219,1.3200000524520874,1.5,1.6100000143051147,nan,nan,nan,0.06666666269302368,0.09876542538404465,0.1308641880750656,9.0865314e-17,1.780534e-17,5.103261942765484,True,7.8516171,0.35799514,2.6111485e-17,2.8477876e-18,9.169042312003887,True,5.8867022,0.56847008,nan,nan,0.0,False,4.7329847e-17,3.6337026e-18,13.025239599960658,True,5.5470789,0.21700709,5.6829247e-17,4.2889702e-18,13.250091362257542,True,5.5511867,0.21716779,nan,nan,0.

In [9]:
N = None

# 1. config: SAME binning as your real galaxy run (virial), just a separate
#    catalog/output_dir so the FITS/cache doesn't collide with the real one.
sky_cfg = replace(cfg,                   # <-- your real galaxy cfg, NOT star_cfg
    catalog="RANDOM_SKY",
    output_dir="./outputs_starpsf",
    num_gal         = N,
    field           = field_select,
    catalog_path    = "catalogs/lya_halo_catalog_cuts/ALL_haoiii_sn5_mosdef_catalog.txt",
    bins      = [0, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20],
    bin_mode  = "virial",
    mask_method     = "image",
    mask_percentile = 90,
    bg_inner_arcsec = 57.0,
    bg_outer_arcsec = 63.0,
    min_bg_fibers   = 25,
    smooth_bg       = True,
    smoothing_values = [200, 20, 300],
    fiber_combine_method   = "biweight",
    rest_delta      = 0.2,
    rest_wave_min   = 1100,
    rest_wave_max   = 1400,
    rest_density    = True,
    cut_radial_bin  = 5,
    min_good_wave   = 100,
    combine_wave_block = 32)

# 2. random in-footprint positions, avoiding real sources, (z,mass) from the
#    real run-table. source_table = the galaxy run-table that built `stacks`.
sky_tbl = starpsf.make_random_sky_table(
    sky_cfg, n=table_len,             # same number as catalog
    source_table=gal_run_table,       # our real galaxy table (has RA/DEC/z/mass)
    avoid_radius_arcsec=10.0,
    ra_bounds=None,
    dec_bounds=None)

In [33]:
sky_tbl_cut = sky_tbl[:150]

In [10]:
# 3. the one expensive extraction — mask='auto' (reject real sources)
sky_prod = starpsf.run_positions_extract(
    sky_cfg, sky_tbl, mask="auto", use_cache=True, write=True)

Loaded AEGIS: 141 exposures, 34944 fibers, 1036 wavelength pixels


the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]


Same RA and bad_fiber_mask shape? True
Bad fibers: 498264
Good fibers: 4428840
Fraction bad: 0.101


 90%|█████████ | 224/248 [00:05<00:00, 46.42it/s]

Capping logMstar 11.01 -> 11.00, z=2.296


 95%|█████████▌| 236/248 [35:21<22:16, 111.36s/it]

Capping logMstar 11.18 -> 11.00, z=2.114


100%|██████████| 248/248 [1:29:29<00:00, 21.65s/it] 


wrote ./outputs_starpsf/galaxy_catc1488c_b8_8c91de_AEGIS_RANDOM_SKY_virial_biweight_image_bg57-63.fits
extracted 248 positions, mask='auto', bin_mode=virial
